# Capstone Project: Hybrid NCF — Sistem Rekomendasi Makanan Aman

Notebook ini melatih model **Hybrid Neural Collaborative Filtering (NCF)** untuk memprediksi
**severity interaksi obat-makanan (skala 0-5)**, yang digunakan sebagai backbone
**sistem rekomendasi makanan aman** pada platform Jivara.

**Alur Sistem Rekomendasi:**
1. Pasien menginput obat yang sedang dikonsumsi
2. Obat di-mapping ke kategori interaksi (14 kategori farmakologis)
3. Model memprediksi **skor severity** interaksi untuk setiap makanan terhadap kategori obat
4. Makanan diranking berdasarkan severity terendah → rekomendasi top-N makanan aman

**Komponen Kustom TensorFlow/Keras:**
- `InteractionEmbeddingLayer` — Custom Layer: embedding + element-wise multiply fusion
- `MedicalSafetyLoss` — Custom Loss: asymmetric MSE, penalti lebih berat untuk under-prediction
- `SafetyThresholdMonitor` — Custom Callback: early stopping saat target MAE tercapai

**Dataset:**
- `drug_food_interactions.csv` — 854 pasangan makanan-obat dengan severity 0-5 (ground truth)
- `food_to_ingredient_kb.json` — 61 kelas makanan YOLO + komposisi bahan
- `obat_bpom_cleaned_full.csv` — 23.682 produk obat BPOM (2.475 komposisi unik)

In [1]:
import os, json, pickle
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, losses, regularizers
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error,
    accuracy_score, classification_report,
    precision_score, recall_score, f1_score
)
from collections import Counter

print("TensorFlow:", tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)

TensorFlow: 2.20.0


## 1. Data Loading

Memuat tiga sumber data:
1. **drug_food_interactions.csv** — Ground truth: 854 pasangan makanan-obat dengan severity 0-5
2. **food_to_ingredient_kb.json** — 61 kelas makanan YOLO beserta daftar bahan masing-masing
3. **obat_bpom_cleaned_full.csv** — 23.682 produk obat terdaftar BPOM dengan komposisi zat aktif

In [2]:
# Ground truth interaksi obat-makanan
df_interactions = pd.read_csv('../data/drug_food_interactions.csv')

# Komposisi bahan makanan
with open('../data/food_to_ingredient_kb.json', 'r', encoding='utf-8') as f:
    food_kb = json.load(f)
food_to_ingredients = food_kb['food_to_ingredients']

# Database obat BPOM
df_bpom = pd.read_csv('../data/obat_bpom_cleaned_full.csv')

print(f"Ground truth interaksi : {len(df_interactions)} pasangan")
print(f"Kelas makanan YOLO     : {len(food_to_ingredients)}")
print(f"Kategori obat          : {df_interactions['drug_category'].nunique()}")
print(f"Produk obat BPOM       : {len(df_bpom)}")
print(f"Komposisi unik BPOM    : {df_bpom['Komposisi'].nunique()}")

print(f"\nDistribusi severity:")
for sev in sorted(df_interactions['severity'].unique()):
    n = len(df_interactions[df_interactions['severity'] == sev])
    labels = {0: "aman", 1: "minimal", 2: "ringan", 3: "sedang", 4: "signifikan", 5: "berat"}
    print(f"  Severity {sev} ({labels.get(sev, '?'):11s}): {n:4d} ({n/len(df_interactions)*100:.1f}%)")

Ground truth interaksi : 854 pasangan
Kelas makanan YOLO     : 61
Kategori obat          : 14
Produk obat BPOM       : 23682
Komposisi unik BPOM    : 2477

Distribusi severity:
  Severity 0 (aman       ):  623 (73.0%)
  Severity 1 (minimal    ):   15 (1.8%)
  Severity 2 (ringan     ):   16 (1.9%)
  Severity 3 (sedang     ):  103 (12.1%)
  Severity 4 (signifikan ):   89 (10.4%)
  Severity 5 (berat      ):    8 (0.9%)


## 2. Knowledge Base Kategori Obat

14 kategori obat farmakologis digunakan untuk **mapping nama obat pasien ke kategori interaksi**.
Setiap kategori berisi:
- **keywords**: nama zat aktif untuk matching terhadap komposisi obat BPOM
- **mechanism**: penjelasan mekanisme interaksi

Mapping ini diperlukan saat inference: pasien menginput "WARFARIN" → sistem mapping ke kategori "antikoagulan" → model prediksi severity untuk semua makanan terhadap kategori tersebut.

In [3]:
DRUG_CATEGORIES = {
    "antikoagulan": {
        "keywords": ["WARFARIN", "CLOPIDOGREL", "HEPARIN", "ENOXAPARIN",
                     "RIVAROXABAN", "APIXABAN", "TICAGRELOR", "ACENOCOUMAROL"],
        "mechanism": "Meningkatkan efek pengencer darah / antagonis vitamin K",
    },
    "antidiabetes": {
        "keywords": ["METFORMIN", "GLIBENCLAMIDE", "GLIMEPIRIDE", "GLIPIZIDE",
                     "GLICLAZIDE", "INSULIN", "ACARBOSE", "PIOGLITAZONE",
                     "SITAGLIPTIN", "VILDAGLIPTIN"],
        "mechanism": "Makanan tinggi gula mengganggu kontrol glikemik",
    },
    "ace_arb": {
        "keywords": ["CAPTOPRIL", "ENALAPRIL", "LISINOPRIL", "RAMIPRIL",
                     "LOSARTAN", "VALSARTAN", "IRBESARTAN", "CANDESARTAN",
                     "TELMISARTAN", "PERINDOPRIL"],
        "mechanism": "Makanan tinggi kalium meningkatkan risiko hiperkalemia",
    },
    "ccb": {
        "keywords": ["AMLODIPINE", "NIFEDIPINE", "DILTIAZEM", "VERAPAMIL",
                     "FELODIPINE"],
        "mechanism": "Jeruk/grapefruit menghambat CYP3A4, meningkatkan kadar obat",
    },
    "statin": {
        "keywords": ["SIMVASTATIN", "ATORVASTATIN", "LOVASTATIN",
                     "ROSUVASTATIN", "PRAVASTATIN", "FLUVASTATIN"],
        "mechanism": "CYP3A4 inhibitor dan lemak tinggi meningkatkan absorpsi berlebih",
    },
    "antibiotik_tetrasiklin": {
        "keywords": ["DOXYCYCLINE", "TETRACYCLINE", "MINOCYCLINE",
                     "OXYTETRACYCLINE"],
        "mechanism": "Kalsium dan mineral mengikat antibiotik, mengurangi absorpsi",
    },
    "antibiotik_fluorokuinolon": {
        "keywords": ["CIPROFLOXACIN", "LEVOFLOXACIN", "MOXIFLOXACIN",
                     "OFLOXACIN", "NORFLOXACIN"],
        "mechanism": "Kation divalen (Ca, Mg, Fe) mengurangi absorpsi antibiotik",
    },
    "maoi": {
        "keywords": ["SELEGILINE", "MOCLOBEMIDE", "LINEZOLID",
                     "TRANYLCYPROMINE", "PHENELZINE", "RASAGILINE"],
        "mechanism": "Tyramine dalam makanan fermentasi menyebabkan krisis hipertensi",
    },
    "tiroid": {
        "keywords": ["LEVOTHYROXINE", "LIOTHYRONINE", "THYROXINE",
                     "LEVOTIROKSIN"],
        "mechanism": "Kedelai dan kalsium mengganggu absorpsi hormon tiroid",
    },
    "nsaid": {
        "keywords": ["IBUPROFEN", "DIKLOFENAK", "DICLOFENAC", "MELOXICAM",
                     "PIROXICAM", "KETOROLAC", "NAPROXEN",
                     "ASAM MEFENAMAT", "INDOMETASIN", "CELECOXIB"],
        "mechanism": "Asam memperburuk iritasi lambung yang disebabkan NSAID",
    },
    "antikonvulsan": {
        "keywords": ["PHENYTOIN", "FENITOIN", "CARBAMAZEPINE",
                     "KARBAMAZEPIN", "VALPROIC", "PHENOBARBITAL"],
        "mechanism": "Kalsium dan protein tinggi mengubah absorpsi antikonvulsan",
    },
    "glikosida_jantung": {
        "keywords": ["DIGOXIN", "DIGOKSIN"],
        "mechanism": "Perubahan kadar kalium mempengaruhi toksisitas digitalis",
    },
    "xantin": {
        "keywords": ["THEOPHYLLINE", "TEOFILIN", "AMINOPHYLLINE", "AMINOFILIN"],
        "mechanism": "Kafein berkompetisi; lemak tinggi mengubah farmakokinetik",
    },
    "imunosupresan": {
        "keywords": ["CYCLOSPORINE", "SIKLOSPORIN", "TACROLIMUS",
                     "SIROLIMUS", "EVEROLIMUS", "MYCOPHENOLATE"],
        "mechanism": "CYP3A4 inhibitor meningkatkan kadar dan toksisitas obat",
    },
}

drug_cat_names = sorted(df_interactions['drug_category'].unique())
food_names = sorted(df_interactions['food_class'].unique())

print(f"Total kategori obat: {len(drug_cat_names)}")
print(f"Total makanan      : {len(food_names)}")
print(f"\nInteraksi per kategori obat (severity >= 2):")
for cat in drug_cat_names:
    cat_df = df_interactions[df_interactions['drug_category'] == cat]
    n_interact = int((cat_df['severity'] >= 2).sum())
    avg_sev = cat_df[cat_df['severity'] > 0]['severity'].mean()
    print(f"  {cat:30s}: {n_interact:2d}/61 makanan, avg severity {avg_sev:.1f}" if avg_sev > 0
          else f"  {cat:30s}: {n_interact:2d}/61 makanan")

Total kategori obat: 14
Total makanan      : 61

Interaksi per kategori obat (severity >= 2):
  ace_arb                       :  5/61 makanan, avg severity 3.6
  antibiotik_fluorokuinolon     :  8/61 makanan, avg severity 4.0
  antibiotik_tetrasiklin        :  8/61 makanan, avg severity 4.0
  antidiabetes                  : 42/61 makanan, avg severity 3.8
  antikoagulan                  : 44/61 makanan, avg severity 3.4
  antikonvulsan                 :  8/61 makanan, avg severity 3.0
  ccb                           :  0/61 makanan, avg severity 1.0
  glikosida_jantung             :  8/61 makanan, avg severity 3.4
  imunosupresan                 : 41/61 makanan, avg severity 3.1
  maoi                          : 22/61 makanan, avg severity 3.8
  nsaid                         :  4/61 makanan, avg severity 1.9
  statin                        :  7/61 makanan, avg severity 2.2
  tiroid                        : 12/61 makanan, avg severity 3.3
  xantin                        :  7/61 makanan,

In [4]:
# Gunakan data dari CSV sebagai DataFrame utama
df = df_interactions[['food_class', 'drug_category', 'severity']].copy()
df.columns = ['food_name', 'drug_category', 'severity']

# Normalisasi severity ke range 0-1 untuk training (sigmoid output)
df['severity_norm'] = df['severity'] / 5.0

# Buat risk category untuk stratified split dan evaluasi
def severity_to_risk(sev):
    if sev == 0: return "aman"
    if sev <= 2: return "ringan"
    if sev <= 3: return "sedang"
    return "tinggi"

df['risk_category'] = df['severity'].apply(severity_to_risk)

print("=== Dataset Ground Truth ===")
print(f"Total pasangan  : {len(df)}")
print(f"Severity range  : {df['severity'].min()} - {df['severity'].max()}")
print(f"Severity mean   : {df['severity'].mean():.2f}")
print(f"\nDistribusi risk category:")
for cat in ["aman", "ringan", "sedang", "tinggi"]:
    n = int((df['risk_category'] == cat).sum())
    print(f"  {cat:8s}: {n:4d} ({n/len(df)*100:.1f}%)")

=== Dataset Ground Truth ===
Total pasangan  : 854
Severity range  : 0 - 5
Severity mean   : 0.88

Distribusi risk category:
  aman    :  623 (73.0%)
  ringan  :   31 (3.6%)
  sedang  :  103 (12.1%)
  tinggi  :   97 (11.4%)


## 3. Feature Engineering

Setiap makanan direpresentasikan oleh dua jenis fitur:
1. **food_id** — Indeks integer untuk embedding layer (collaborative signal)
2. **Ingredient multi-hot vector** — Vektor biner 25-dimensi menunjukkan keberadaan bahan farmakologis aktif (content signal)

Target: **severity ternormalisasi** (0-1), di mana 0 = aman, 1 = severity 5 (berat).

In [ ]:
# ============================================================
# A. Daftar Bahan Farmakologis Aktif (content feature vector)
# ============================================================
ACTIVE_KEYWORDS = [
    "bawang putih", "jahe", "kunyit", "lengkuas",
    "kangkung", "bayam", "kemangi", "daun bawang",
    "gula pasir", "gula merah", "gula aren", "kental manis", "madu",
    "pisang", "kentang",
    "santan",
    "jeruk",
    "susu",
    "kecap", "petis", "terasi",
    "tempe", "tahu",
    "cuka", "kopi",
]
print(f"Jumlah bahan aktif yang di-track: {len(ACTIVE_KEYWORDS)}")

# ============================================================
# B. Generate ingredient multi-hot features per makanan
# ============================================================
def get_ingredient_features(food_ingredients, keywords):
    """Generate multi-hot vector: 1 jika keyword ditemukan di daftar bahan."""
    return [1.0 if any(kw.lower() in ingr.lower() for ingr in food_ingredients)
            else 0.0 for kw in keywords]

ingredient_feature_map = {}
for food in food_names:
    ingredient_feature_map[food] = get_ingredient_features(
        food_to_ingredients[food], ACTIVE_KEYWORDS
    )

# ============================================================
# C. LabelEncoder + Prepare Arrays
# ============================================================
le_food = LabelEncoder()
le_drug = LabelEncoder()
le_food.fit(food_names)
le_drug.fit(drug_cat_names)

num_foods = len(le_food.classes_)
num_drugs = len(le_drug.classes_)
n_ingredient_features = len(ACTIVE_KEYWORDS)

df['food_id'] = le_food.transform(df['food_name'])
df['drug_id'] = le_drug.transform(df['drug_category'])

X_food = df['food_id'].values
X_drug = df['drug_id'].values
X_ingredients = np.array([
    ingredient_feature_map[food] for food in df['food_name']
], dtype=np.float32)

# Target = severity ternormalisasi (0-1)
y = df['severity_norm'].values.astype(np.float32)

# Stratification bins untuk K-Fold (berdasarkan risk category)
y_strat = df['risk_category'].values

# ============================================================
# D. Sample Weights — handle class imbalance
# ============================================================
severity_counts = df['severity'].value_counts()
n_samples = len(df)
n_severity_classes = len(severity_counts)
class_weight_map = {
    sev: np.sqrt(n_samples / (n_severity_classes * count))
    for sev, count in severity_counts.items()
}
sample_weights = df['severity'].map(class_weight_map).values.astype(np.float32)

print(f"\nDataset siap training:")
print(f"  Makanan unik          : {num_foods}")
print(f"  Kategori obat unik    : {num_drugs}")
print(f"  Fitur ingredient      : {n_ingredient_features}-dim")
print(f"  Total sampel          : {len(y)}")
print(f"  Target range          : [{y.min():.2f}, {y.max():.2f}] (severity/5)")
print(f"  Target mean           : {y.mean():.4f} (severity {y.mean()*5:.2f}/5)")

print(f"\nSample weights per severity:")
for sev in sorted(class_weight_map):
    print(f"  Severity {sev}: weight={class_weight_map[sev]:.3f} (n={severity_counts[sev]})")

## 4. Custom Components

Tiga komponen kustom TensorFlow/Keras:

1. **InteractionEmbeddingLayer** — Custom Layer: embed food_id dan drug_id ke ruang vektor, gabungkan via element-wise multiply untuk menangkap pola interaksi non-linear.

2. **MedicalSafetyLoss** — Custom Loss (asymmetric MSE): penalti lebih berat untuk under-prediction (prediksi severity rendah padahal sebenarnya tinggi). Under-prediction lebih berbahaya karena pasien tidak diperingatkan.

3. **SafetyThresholdMonitor** — Custom Callback: early stopping saat validation MAE mencapai target.

In [6]:
class InteractionEmbeddingLayer(layers.Layer):
    """
    Custom Layer — Embedding + Element-wise Multiply.
    Menerima (food_id, drug_id), embed ke vektor, multiply untuk
    menghasilkan representasi interaksi.
    """
    def __init__(self, num_foods, num_drugs, embedding_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.num_foods = num_foods
        self.num_drugs = num_drugs
        self.embedding_dim = embedding_dim
        self.food_embedding = layers.Embedding(
            input_dim=num_foods, output_dim=embedding_dim, name="food_emb")
        self.drug_embedding = layers.Embedding(
            input_dim=num_drugs, output_dim=embedding_dim, name="drug_emb")
        self.multiply_layer = layers.Multiply(name="interaction_fusion")

    def call(self, inputs):
        food_id, drug_id = inputs[0], inputs[1]
        return self.multiply_layer([
            self.food_embedding(food_id),
            self.drug_embedding(drug_id),
        ])

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_foods": self.num_foods,
            "num_drugs": self.num_drugs,
            "embedding_dim": self.embedding_dim,
        })
        return config


class MedicalSafetyLoss(losses.Loss):
    """
    Custom Loss — Asymmetric MSE.
    Under-prediction (actual > predicted) mendapat penalti lebih besar
    karena melewatkan interaksi berbahaya lebih berisiko daripada false alarm.
    """
    def __init__(self, under_penalty=1.5, name="medical_safety_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.under_penalty = under_penalty

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        error = y_true - y_pred
        # error > 0 berarti under-prediction (actual severity > predicted)
        weight = tf.where(error > 0, self.under_penalty, 1.0)
        return weight * tf.square(error)

    def get_config(self):
        config = super().get_config()
        config.update({"under_penalty": self.under_penalty})
        return config


class SafetyThresholdMonitor(tf.keras.callbacks.Callback):
    """
    Custom Callback — Early stopping saat val_mae <= target.
    """
    def __init__(self, target_mae=0.05):
        super().__init__()
        self.target_mae = target_mae

    def on_epoch_end(self, epoch, logs=None):
        val_mae = logs.get('val_mae')
        if val_mae is not None and val_mae <= self.target_mae:
            print(f"\n[TARGET TERCAPAI] Epoch {epoch+1}: "
                  f"val_MAE = {val_mae:.4f} (<= {self.target_mae})")
            self.model.stop_training = True

print("Custom components defined:")
print("  - InteractionEmbeddingLayer (embed + multiply)")
print("  - MedicalSafetyLoss (asymmetric MSE, under_penalty=1.5)")
print("  - SafetyThresholdMonitor (early stop on MAE)")

Custom components defined:
  - InteractionEmbeddingLayer (embed + multiply)
  - MedicalSafetyLoss (asymmetric MSE, under_penalty=1.5)
  - SafetyThresholdMonitor (early stop on MAE)


## 5. Arsitektur Hybrid NCF (Drug-Aware Content Branch)

```
food_id ──► Embedding(8) ──┐
                            ├── Multiply ──► collab(8) ──┐
drug_id ──► Embedding(8) ──┘                              │
                                                          ├── Concat(40) ──► Dense(64) ──► Dense(32) ──► Severity (0-1)
drug_id ──► Embedding(8) ──┐                              │
                            ├── Concat(33) ──► Dense(32) ─┘
ingredient(25) ────────────┘
```

**Perbedaan dengan arsitektur sebelumnya:**
- **Drug-aware content branch**: drug embedding di-concatenate dengan ingredient features sebelum Dense layer, sehingga model tahu *kategori obat mana* yang sedang dievaluasi terhadap bahan makanan
- **Embedding dim 8** (dari 16): dataset kecil (854 sampel) tidak butuh embedding besar
- **Tanpa L2 regularization**: model sebelumnya under-fitting (prediksi mean), bukan over-fitting
- **Dropout 0.2** (dari 0.3): mengurangi regularisasi berlebih

In [ ]:
def build_hybrid_ncf(num_foods, num_drugs, embedding_dim=8, n_features=25):
    input_food = layers.Input(shape=(1,), name="input_food_id")
    input_drug = layers.Input(shape=(1,), name="input_drug_id")
    input_features = layers.Input(shape=(n_features,), name="input_ingredient_features")

    # === Collaborative branch: food × drug embedding ===
    interaction = InteractionEmbeddingLayer(
        num_foods, num_drugs, embedding_dim
    )([input_food, input_drug])
    collab_flat = layers.Flatten()(interaction)

    # === Content branch (DRUG-AWARE): drug context + ingredient features ===
    drug_context_emb = layers.Embedding(
        input_dim=num_drugs, output_dim=embedding_dim, name="drug_context_emb"
    )(input_drug)
    drug_context_flat = layers.Flatten()(drug_context_emb)
    content_input = layers.Concatenate(name="content_concat")(
        [drug_context_flat, input_features]
    )
    content = layers.Dense(32, activation="relu", name="ingredient_encoder")(content_input)

    # === Merge + prediction (LINEAR output untuk regresi) ===
    merged = layers.Concatenate(name="hybrid_merge")([collab_flat, content])
    x = layers.Dense(64, activation="relu")(merged)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation="relu")(x)
    output = layers.Dense(1, activation="linear", name="severity_output")(x)

    return models.Model(
        inputs=[input_food, input_drug, input_features],
        outputs=output,
        name="Jivara_HybridNCF_Recommender",
    )

model = build_hybrid_ncf(num_foods, num_drugs, n_features=n_ingredient_features)
model.summary()

## 6. Stratified K-Fold Cross Validation

Validasi dengan 5-fold stratified CV (stratifikasi berdasarkan risk category). Metrik utama:
- **MAE** — mean absolute error severity (skala 0-5)
- **RMSE** — root mean squared error
- **Risk Accuracy** — akurasi klasifikasi risk category (aman/ringan/sedang/tinggi)

In [ ]:
def pred_to_risk(severity_pred):
    """Konversi predicted severity (0-5) ke risk category."""
    if severity_pred < 1.0: return "aman"
    if severity_pred < 2.5: return "ringan"
    if severity_pred < 3.5: return "sedang"
    return "tinggi"

n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
fold_results = []

print(f"Menjalankan {n_folds}-Fold Stratified Cross Validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_food, y_strat)):
    print(f"--- Fold {fold+1}/{n_folds} ---")

    fold_model = build_hybrid_ncf(num_foods, num_drugs, n_features=n_ingredient_features)
    fold_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
        loss='mse',
        metrics=['mae'],
    )

    fold_model.fit(
        x={"input_food_id": X_food[train_idx],
           "input_drug_id": X_drug[train_idx],
           "input_ingredient_features": X_ingredients[train_idx]},
        y=y[train_idx],
        sample_weight=sample_weights[train_idx],
        validation_data=(
            {"input_food_id": X_food[val_idx],
             "input_drug_id": X_drug[val_idx],
             "input_ingredient_features": X_ingredients[val_idx]},
            y[val_idx],
        ),
        epochs=500,
        batch_size=32,
        callbacks=[
            SafetyThresholdMonitor(target_mae=0.05),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_mae', patience=50, restore_best_weights=True
            ),
        ],
        verbose=0,
    )

    # Prediksi, clip ke [0,1], dan denormalisasi ke skala 0-5
    val_preds_norm = fold_model.predict({
        "input_food_id": X_food[val_idx],
        "input_drug_id": X_drug[val_idx],
        "input_ingredient_features": X_ingredients[val_idx],
    }, verbose=0).flatten()
    val_preds_norm = np.clip(val_preds_norm, 0, 1)
    val_preds_sev = val_preds_norm * 5.0
    val_true_sev = y[val_idx] * 5.0

    mae = mean_absolute_error(val_true_sev, val_preds_sev)
    rmse = np.sqrt(mean_squared_error(val_true_sev, val_preds_sev))

    # Risk category accuracy
    true_risk = [severity_to_risk(s) for s in val_true_sev]
    pred_risk = [pred_to_risk(s) for s in val_preds_sev]
    risk_acc = accuracy_score(true_risk, pred_risk)

    fold_results.append({'mae': mae, 'rmse': rmse, 'risk_acc': risk_acc})
    print(f"  MAE={mae:.3f} | RMSE={rmse:.3f} | Risk Acc={risk_acc:.3f}\n")

print("=" * 60)
print("HASIL CROSS VALIDATION (Hybrid NCF Severity Regression)")
print("=" * 60)
for metric in ['mae', 'rmse', 'risk_acc']:
    vals = [r[metric] for r in fold_results]
    label = {'mae': 'MAE (0-5)', 'rmse': 'RMSE (0-5)', 'risk_acc': 'Risk Accuracy'}[metric]
    print(f"  {label:15s}: {np.mean(vals):.4f} (+/- {np.std(vals):.4f})")
print("=" * 60)

## 7. Training Model Final & Ekspor

Latih model final dengan seluruh data, lalu ekspor model dan artifacts ke `models/`.

In [9]:
final_model = build_hybrid_ncf(num_foods, num_drugs, n_features=n_ingredient_features)
final_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.003),
    loss=MedicalSafetyLoss(under_penalty=1.5),
    metrics=['mae'],
)

class ProgressPrinter(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 200 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1:4d}: loss={logs.get('loss',0):.4f}, "
                  f"MAE={logs.get('mae',0):.4f}")

print("Training model final pada seluruh dataset...\n")
history = final_model.fit(
    x={"input_food_id": X_food,
       "input_drug_id": X_drug,
       "input_ingredient_features": X_ingredients},
    y=y,
    epochs=1000,
    batch_size=32,
    callbacks=[ProgressPrinter()],
    verbose=0,
)

# === Simpan model & artifacts ===
os.makedirs('../models', exist_ok=True)

model_path = '../models/drug_interaction_model.keras'
final_model.save(model_path)
print(f"\nModel disimpan: {model_path}")

artifacts = {
    'le_food_classes': list(le_food.classes_),
    'le_drug_classes': list(le_drug.classes_),
    'active_keywords': ACTIVE_KEYWORDS,
    'ingredient_feature_map': ingredient_feature_map,
    'drug_categories': {k: v for k, v in DRUG_CATEGORIES.items()},
}
artifacts_path = '../models/recommender_artifacts.pkl'
with open(artifacts_path, 'wb') as f:
    pickle.dump(artifacts, f)
print(f"Artifacts disimpan: {artifacts_path}")

Training model final pada seluruh dataset...

  Epoch    1: loss=0.1457, MAE=0.3186
  Epoch  200: loss=0.1193, MAE=0.2761
  Epoch  400: loss=0.1190, MAE=0.2741
  Epoch  600: loss=0.1189, MAE=0.2740
  Epoch  800: loss=0.1189, MAE=0.2750
  Epoch 1000: loss=0.1189, MAE=0.2740

Model disimpan: ../models/drug_interaction_model.keras
Artifacts disimpan: ../models/recommender_artifacts.pkl


## 8. Sistem Rekomendasi Makanan & Evaluasi

**Alur Rekomendasi:**
1. Pasien menginput nama obat / komposisi zat aktif
2. Sistem mapping obat ke kategori interaksi (keyword matching terhadap BPOM)
3. Model prediksi **severity** interaksi untuk 61 makanan (skala 0-5)
4. Makanan diurutkan dari severity terendah (paling aman)
5. Top-N makanan aman direkomendasikan, makanan dengan severity >= 3 ditandai "hindari"

In [10]:
def map_drug_to_categories(drug_name, bpom_df=None, drug_categories=DRUG_CATEGORIES):
    """Map nama obat/komposisi ke kategori interaksi."""
    drug_upper = drug_name.strip().upper()
    matched = []
    for cat, info in drug_categories.items():
        for kw in info['keywords']:
            if kw in drug_upper:
                matched.append(cat)
                break
    if not matched and bpom_df is not None:
        bpom_match = bpom_df[
            bpom_df['Nama Produk'].str.upper().str.contains(drug_upper, na=False)
        ]
        if not bpom_match.empty:
            composition = str(bpom_match.iloc[0]['Komposisi']).upper()
            for cat, info in drug_categories.items():
                for kw in info['keywords']:
                    if kw in composition:
                        matched.append(cat)
                        break
    return list(set(matched))


def recommend_foods(medications, model, le_food, le_drug,
                    ingredient_feature_map, drug_categories,
                    bpom_df=None, top_n=10):
    """Rekomendasi makanan aman berdasarkan predicted severity."""
    all_cats = set()
    med_cat_map = {}
    for med in medications:
        cats = map_drug_to_categories(med, bpom_df, drug_categories)
        med_cat_map[med] = cats if cats else ['(tidak ditemukan)']
        all_cats.update(cats)

    if not all_cats:
        return {"status": "no_interaction_data",
                "message": "Obat tidak ditemukan. Semua makanan dianggap aman.",
                "recommended_foods": list(le_food.classes_)[:top_n]}

    all_foods = list(le_food.classes_)
    food_scores = {}
    for food in all_foods:
        food_id = le_food.transform([food])[0]
        features = np.array([ingredient_feature_map[food]], dtype=np.float32)
        max_severity = 0.0
        worst_cat = None
        for cat in all_cats:
            drug_id = le_drug.transform([cat])[0]
            pred_norm = float(model.predict({
                "input_food_id": np.array([food_id]),
                "input_drug_id": np.array([drug_id]),
                "input_ingredient_features": features,
            }, verbose=0)[0][0])
            severity = pred_norm * 5.0
            if severity > max_severity:
                max_severity = severity
                worst_cat = cat

        risk = pred_to_risk(max_severity)
        food_scores[food] = {
            "predicted_severity": round(max_severity, 2),
            "risk_level": risk,
            "worst_category": worst_cat,
        }

    sorted_foods = sorted(food_scores.items(),
                          key=lambda x: x[1]['predicted_severity'])
    safe = [f for f, s in sorted_foods if s['risk_level'] in ('aman', 'ringan')]
    avoid = [f for f, s in sorted_foods if s['risk_level'] in ('sedang', 'tinggi')]

    return {
        "patient_medications": medications,
        "matched_categories": med_cat_map,
        "total_foods": len(all_foods),
        "safe_count": len(safe), "avoid_count": len(avoid),
        "recommended_foods": [{"food": f, **food_scores[f]} for f, _ in sorted_foods[:top_n]],
        "foods_to_avoid": [{"food": f, **food_scores[f]} for f in [f for f,_ in reversed(sorted_foods) if food_scores[f]['risk_level'] in ('sedang', 'tinggi')]],
    }

In [11]:
# === DEMO 1: Pasien Warfarin ===
print("=" * 70)
print("  DEMO 1: Rekomendasi untuk pasien WARFARIN (antikoagulan)")
print("=" * 70)
r1 = recommend_foods(["WARFARIN"], final_model, le_food, le_drug,
                     ingredient_feature_map, DRUG_CATEGORIES, df_bpom, 10)
print(f"Obat      : {r1['patient_medications']}")
print(f"Kategori  : {r1['matched_categories']}")
print(f"Aman: {r1['safe_count']} | Hindari: {r1['avoid_count']}")
print(f"\nTop 10 Rekomendasi Makanan Aman:")
for i, f in enumerate(r1['recommended_foods'], 1):
    print(f"  {i:2d}. {f['food']:25s} severity={f['predicted_severity']:.2f}/5 [{f['risk_level']}]")
if r1['foods_to_avoid']:
    print(f"\nMakanan yang Harus Dihindari:")
    for f in r1['foods_to_avoid'][:10]:
        print(f"  x {f['food']:25s} severity={f['predicted_severity']:.2f}/5 [{f['risk_level']}] ({f['worst_category']})")

print("\n")

# === DEMO 2: Pasien Metformin + Simvastatin ===
print("=" * 70)
print("  DEMO 2: Rekomendasi untuk pasien METFORMIN + SIMVASTATIN")
print("=" * 70)
r2 = recommend_foods(["METFORMIN", "SIMVASTATIN"], final_model, le_food, le_drug,
                     ingredient_feature_map, DRUG_CATEGORIES, df_bpom, 10)
print(f"Obat      : {r2['patient_medications']}")
print(f"Kategori  : {r2['matched_categories']}")
print(f"Aman: {r2['safe_count']} | Hindari: {r2['avoid_count']}")
print(f"\nTop 10 Rekomendasi Makanan Aman:")
for i, f in enumerate(r2['recommended_foods'], 1):
    print(f"  {i:2d}. {f['food']:25s} severity={f['predicted_severity']:.2f}/5 [{f['risk_level']}]")
if r2['foods_to_avoid']:
    print(f"\nMakanan yang Harus Dihindari:")
    for f in r2['foods_to_avoid'][:10]:
        print(f"  x {f['food']:25s} severity={f['predicted_severity']:.2f}/5 [{f['risk_level']}] ({f['worst_category']})")

  DEMO 1: Rekomendasi untuk pasien WARFARIN (antikoagulan)
Obat      : ['WARFARIN']
Kategori  : {'WARFARIN': ['antikoagulan']}
Aman: 61 | Hindari: 0

Top 10 Rekomendasi Makanan Aman:
   1. nugget                    severity=0.50/5 [aman]
   2. tahu-telur                severity=0.74/5 [aman]
   3. steak                     severity=0.87/5 [aman]
   4. bika-ambon                severity=0.88/5 [aman]
   5. ayam-bumbu-rujak          severity=0.92/5 [aman]
   6. kiwi                      severity=0.92/5 [aman]
   7. apel                      severity=0.93/5 [aman]
   8. kue-lumpur                severity=0.94/5 [aman]
   9. kolak                     severity=0.95/5 [aman]
  10. burger                    severity=0.96/5 [aman]


  DEMO 2: Rekomendasi untuk pasien METFORMIN + SIMVASTATIN
Obat      : ['METFORMIN', 'SIMVASTATIN']
Kategori  : {'METFORMIN': ['antidiabetes'], 'SIMVASTATIN': ['statin']}
Aman: 61 | Hindari: 0

Top 10 Rekomendasi Makanan Aman:
   1. sate-lilit                severi

In [12]:
# === Evaluasi Model Final ===
all_preds_norm = final_model.predict({
    "input_food_id": X_food,
    "input_drug_id": X_drug,
    "input_ingredient_features": X_ingredients,
}, verbose=0).flatten()

# Denormalisasi ke skala 0-5
all_preds_sev = all_preds_norm * 5.0
all_true_sev = y * 5.0

final_mae = mean_absolute_error(all_true_sev, all_preds_sev)
final_rmse = np.sqrt(mean_squared_error(all_true_sev, all_preds_sev))

# Risk category classification
true_risk = [severity_to_risk(s) for s in all_true_sev]
pred_risk = [pred_to_risk(s) for s in all_preds_sev]
risk_acc = accuracy_score(true_risk, pred_risk)

print("=" * 60)
print("  LAPORAN EVALUASI — HYBRID NCF RECOMMENDER (MODEL FINAL)")
print("=" * 60)
s_mae = "LULUS" if final_mae <= 0.50 else "BELUM"
s_rmse = "LULUS" if final_rmse <= 0.80 else "BELUM"
s_risk = "LULUS" if risk_acc >= 0.85 else "BELUM"
print(f"  MAE (0-5)       : {final_mae:.4f}  [{s_mae} - target <= 0.50]")
print(f"  RMSE (0-5)      : {final_rmse:.4f}  [{s_rmse} - target <= 0.80]")
print(f"  Risk Accuracy   : {risk_acc*100:.1f}%  [{s_risk} - target >= 85%]")
print("=" * 60)

print("\nRisk Category Classification Report:")
print(classification_report(
    true_risk, pred_risk,
    labels=["aman", "ringan", "sedang", "tinggi"],
    zero_division=0,
))

# Scatter analysis: actual vs predicted
print("Prediksi per severity level:")
for sev in sorted(df['severity'].unique()):
    mask = (all_true_sev == sev)
    if mask.sum() > 0:
        preds = all_preds_sev[mask]
        print(f"  Severity {sev}: n={mask.sum():3d}, "
              f"pred mean={preds.mean():.2f}, "
              f"pred range=[{preds.min():.2f}, {preds.max():.2f}]")

  LAPORAN EVALUASI — HYBRID NCF RECOMMENDER (MODEL FINAL)
  MAE (0-5)       : 1.3905  [BELUM - target <= 0.50]
  RMSE (0-5)      : 1.5257  [BELUM - target <= 0.80]
  Risk Accuracy   : 27.9%  [BELUM - target >= 85%]

Risk Category Classification Report:
              precision    recall  f1-score   support

        aman       0.81      0.34      0.48       623
      ringan       0.04      0.81      0.08        31
      sedang       0.00      0.00      0.00       103
      tinggi       0.00      0.00      0.00        97

    accuracy                           0.28       854
   macro avg       0.21      0.29      0.14       854
weighted avg       0.59      0.28      0.35       854

Prediksi per severity level:
  Severity 0: n=623, pred mean=1.16, pred range=[0.47, 1.88]
  Severity 1: n= 15, pred mean=1.25, pred range=[0.76, 1.52]
  Severity 2: n= 16, pred mean=1.26, pred range=[0.86, 1.84]
  Severity 3: n=103, pred mean=1.24, pred range=[0.50, 1.85]
  Severity 4: n= 89, pred mean=1.29, pr